# Feature Engineering

## Spotify Track Popularity Analytics

### Purpose

This notebook prepares the cleaned Spotify dataset for machine learning by creating and selecting model-ready features from the existing track, album, genre, artist, and release information.

The feature engineering process is designed to support the predictive modelling framework established in `00_business_background.ipynb`.

The primary regression target is:

* `track_popularity`

A secondary binary target will also be created to classify tracks as **high popularity** or **lower popularity**. The classification threshold will be determined using a reproducible rule based on the observed `track_popularity` distribution.

## Predictor Strategy

Two feature sets will be prepared and compared.

### Model A: Intrinsic Track and Release Features

Model A addresses the main predictive research question by testing whether track popularity can be predicted without relying on the artist's existing popularity or follower count.

Potential feature groups include:

* Track duration
* Explicit content
* Genre
* Album type
* Album size
* Track position
* Release period

### Model B: Full Information Benchmark

Model B will use the same intrinsic track and release features as Model A and additionally include:

* Artist popularity
* Artist followers

This model will serve as a benchmark to measure how much predictive performance changes when information about the artist's established market position is available.

Artist popularity will be interpreted cautiously because Spotify's artist popularity measure is related to the popularity of the artist's tracks.

## Feature Engineering Objectives

This notebook will:

1. Inspect the cleaned dataset and confirm available variables.
2. Create appropriate model-ready features.
3. Transform variables where necessary.
4. Create the high-popularity classification target.
5. Identify numerical and categorical predictors.
6. Define the feature sets for Model A and Model B.
7. Exclude identifiers and variables that could cause target leakage.
8. Prepare the feature-engineered dataset for preprocessing and machine learning.

## Modelling Considerations

The feature engineering process will account for:

* Missing or unknown genre values
* Multi-label genre information
* Repeated tracks appearing under different IDs
* Skewed artist follower counts
* Redundancy between release year, release period, and track age
* Target leakage
* Potential class imbalance in the classification target

The actual feature transformations will be determined after inspecting the available columns and distributions in the cleaned dataset.


## 1. Import Libraries

### Purpose

Import the libraries required for data manipulation, numerical calculations, and feature engineering.

Additional preprocessing and machine learning libraries will be introduced later when they are required.


In [1]:
import pandas as pd
import numpy as np

## 2. Load and Inspect the Cleaned Dataset

### Purpose

Load the cleaned Spotify dataset and inspect its structure before creating any new features.

This step confirms the available columns, dataset size, and data types so that feature engineering is based on the actual cleaned data rather than assumed variable names.

The inspection will also help identify which existing variables can be used directly, which require transformation, and which should be excluded from machine learning.


In [6]:
df = pd.read_csv("data/clean/spotify_final.csv")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Shape: (8775, 16)

Columns:
['track_id', 'track_name', 'track_number', 'track_popularity', 'explicit', 'artist_name', 'artist_popularity', 'artist_followers', 'artist_genres', 'album_id', 'album_name', 'album_release_date', 'album_total_tracks', 'album_type', 'track_duration_min', 'release_year']


In [7]:
df.head()

,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min,release_year
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,False,Britney Spears,80.0,17755451.0,pop,325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,58,compilation,3.55,2009
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,False,BUNT.,69.0,293734.0,stutter house,2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single,2.65,2023
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,False,Taylor Swift,100.0,145396321.0,NaN,4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,26,album,3.76,2021
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,True,Billie Eilish,90.0,118692183.0,NaN,0JGOiO34nwfUdDrD612dOp,Happier Than Ever,2021-07-30,16,album,2.64,2021
4,0GLfodYacy3BJE7AI3A8en,Man Down,7,57,False,Rihanna,90.0,68997177.0,NaN,5QG3tjE5L9F6O2vCAPph38,Loud,2010-01-01,13,album,4.45,2010


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8775 entries, 0 to 8774
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   track_id            8775 non-null   str    
 1   track_name          8775 non-null   str    
 2   track_number        8775 non-null   int64  
 3   track_popularity    8775 non-null   int64  
 4   explicit            8775 non-null   bool   
 5   artist_name         8773 non-null   str    
 6   artist_popularity   8772 non-null   float64
 7   artist_followers    8772 non-null   float64
 8   artist_genres       4283 non-null   str    
 9   album_id            8775 non-null   str    
 10  album_name          8775 non-null   str    
 11  album_release_date  8775 non-null   str    
 12  album_total_tracks  8775 non-null   int64  
 13  album_type          8775 non-null   str    
 14  track_duration_min  8775 non-null   float64
 15  release_year        8775 non-null   int64  
dtypes: bool(1), float

## 3. Assess Feature Availability

### Purpose

Evaluate the variables available for machine learning and identify which features can be used directly, which require transformation, and which should be excluded.

The feature selection follows the predictive modelling framework established in `00_business_background.ipynb`.

### Model A: Intrinsic Track and Release Features

The cleaned dataset contains the following potential predictors:

* `track_duration_min`
* `explicit`
* `artist_genres`
* `album_total_tracks`
* `album_type`
* `track_number`
* `release_year`

These variables describe characteristics of the track or its release and do not depend directly on the artist's current popularity or follower count.

### Model B: Full Information Benchmark

Model B will use the same predictors as Model A and additionally include:

* `artist_popularity`
* `artist_followers`

These artist-level variables will be used only as benchmark predictors to determine how much predictive performance improves when information about the artist's established market position is available.

### Target

The primary regression target is:

* `track_popularity`

A binary `high_popularity` target will later be created for the classification analysis.

### Variables Excluded from Prediction

The following variables will not be used directly as predictors:

* `track_id`
* `track_name`
* `artist_name`
* `album_id`
* `album_name`
* `album_release_date`

These variables primarily identify tracks, artists, or albums and are retained for reference rather than predictive modelling.


In [9]:
selected_columns = [
    "track_popularity",
    "track_duration_min",
    "explicit",
    "artist_genres",
    "album_total_tracks",
    "album_type",
    "track_number",
    "release_year",
    "artist_popularity",
    "artist_followers"
]

df[selected_columns].isna().sum()

track_popularity         0
track_duration_min       0
explicit                 0
artist_genres         4492
album_total_tracks       0
album_type               0
track_number             0
release_year             0
artist_popularity        3
artist_followers         3
dtype: int64

In [10]:
for column in selected_columns:
    print(f"{column}: {df[column].nunique()} unique values")

track_popularity: 99 unique values
track_duration_min: 646 unique values
explicit: 2 unique values
artist_genres: 652 unique values
album_total_tracks: 82 unique values
album_type: 3 unique values
track_number: 54 unique values
release_year: 68 unique values
artist_popularity: 96 unique values
artist_followers: 3762 unique values


In [11]:
df["album_type"].value_counts(dropna=False)

album_type
album          6022
single         2245
compilation     508
Name: count, dtype: int64

## 4. Handle Genre Information

### Purpose

Prepare the genre variable for machine learning while preserving tracks with missing genre information.

The original `artist_genres` variable contains a large number of missing values and many unique genre labels. Removing tracks with missing genre information would substantially reduce the dataset and may introduce unnecessary selection bias.

For this reason, missing genre values will be assigned to an `Unknown` category.

The resulting genre variable will then be inspected before deciding whether additional grouping is necessary for machine learning.


In [12]:
df_model = df.copy()

In [13]:
df_model["artist_genres"] = df_model["artist_genres"].fillna("Unknown")

In [14]:
print("Missing genres:", df_model["artist_genres"].isna().sum())
print("Unique genre values:", df_model["artist_genres"].nunique())

Missing genres: 0
Unique genre values: 653


In [15]:
df_model["artist_genres"].value_counts().head(20)

artist_genres
Unknown                                                           4492
soundtrack                                                         346
pop                                                                303
soft pop                                                           209
rap                                                                192
dark r&b                                                           111
k-pop                                                               93
grunge, rock                                                        91
country                                                             90
edm                                                                 82
rap, hip hop                                                        77
country, acoustic country                                           73
art pop, pop                                                        72
r&b                                                            

### 4.1 Create a Primary Genre Feature

The original `artist_genres` variable contains both individual genres and combinations of multiple genres.

Using every unique genre combination as a separate categorical predictor would create a large number of sparse categories. To simplify the feature while retaining meaningful genre information, a new `primary_genre` variable will be created using the first listed genre.

Tracks without available genre information will remain classified as `Unknown`.

This approach provides a consistent categorical feature suitable for the initial machine learning models while reducing the complexity of the original multi-label genre field.


In [16]:
df_model["primary_genre"] = (
    df_model["artist_genres"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [17]:
print("Unique original genre values:", df_model["artist_genres"].nunique())
print("Unique primary genres:", df_model["primary_genre"].nunique())
print("Missing primary genres:", df_model["primary_genre"].isna().sum())

Unique original genre values: 653
Unique primary genres: 279
Missing primary genres: 0


In [18]:
df_model["primary_genre"].value_counts().head(20)

primary_genre
Unknown         4492
soundtrack       353
pop              303
rap              275
country          252
soft pop         224
dark r&b         119
art pop          117
edm              100
grunge            97
k-pop             93
reggaeton         85
emo rap           82
medieval          75
anime             73
r&b               69
nu metal          66
classic rock      64
egyptian pop      59
bedroom pop       52
Name: count, dtype: int64

In [19]:
genre_counts = df_model["primary_genre"].value_counts()

print("Genres with fewer than 10 tracks:", (genre_counts < 10).sum())
print("Genres with fewer than 20 tracks:", (genre_counts < 20).sum())
print("Genres with fewer than 30 tracks:", (genre_counts < 30).sum())
print("Genres with fewer than 50 tracks:", (genre_counts < 50).sum())

Genres with fewer than 10 tracks: 212
Genres with fewer than 20 tracks: 236
Genres with fewer than 30 tracks: 247
Genres with fewer than 50 tracks: 259


### 4.2 Group Rare Primary Genres

The `primary_genre` feature contains 279 categories, but most occur infrequently in the dataset.

Of these categories:

* 212 contain fewer than 10 tracks
* 236 contain fewer than 20 tracks
* 247 contain fewer than 30 tracks
* 259 contain fewer than 50 tracks

Very small categories can create sparse features during one-hot encoding and may make model estimates less stable.

For the initial machine learning models, primary genres represented by fewer than 30 tracks will therefore be grouped into an `Other` category.

The `Unknown` category will remain separate because it represents missing genre information rather than a rare musical genre.

This threshold retains the more frequently represented genre categories while reducing unnecessary categorical complexity.


In [20]:
genre_counts = df_model["primary_genre"].value_counts()

rare_genres = genre_counts[
    (genre_counts < 30) &
    (genre_counts.index != "Unknown")
].index

df_model["genre_grouped"] = df_model["primary_genre"].replace(
    rare_genres,
    "Other"
)

In [21]:
print("Original primary genres:", df_model["primary_genre"].nunique())
print("Grouped genre categories:", df_model["genre_grouped"].nunique())
print("Missing grouped genres:", df_model["genre_grouped"].isna().sum())

Original primary genres: 279
Grouped genre categories: 33
Missing grouped genres: 0


In [22]:
df_model["genre_grouped"].value_counts()

genre_grouped
Unknown               4492
Other                 1245
soundtrack             353
pop                    303
rap                    275
country                252
soft pop               224
dark r&b               119
art pop                117
edm                    100
grunge                  97
k-pop                   93
reggaeton               85
emo rap                 82
medieval                75
anime                   73
r&b                     69
nu metal                66
classic rock            64
egyptian pop            59
bedroom pop             52
industrial metal        49
rage rap                49
latin pop               48
east coast hip hop      43
art rock                41
melodic rap             39
slap house              39
hypertechno             37
indie                   37
hyperpop                34
egyptian hip hop        33
celtic                  31
Name: count, dtype: int64

## 5. Engineer Track and Release Features

### 5.1 Relative Track Position

### Purpose

Create a standardized measure of where a track appears within its album or release.

The existing `track_number` variable provides the absolute position of a track, while `album_total_tracks` provides the total number of tracks on the release. Because releases vary substantially in size, track number alone does not provide a comparable measure of position across albums, singles, and compilations.

A new `relative_track_position` feature will therefore be calculated as:

**Relative Track Position = Track Number / Total Album Tracks**

Values closer to the beginning of the range indicate tracks appearing earlier in a release, while values closer to 1 indicate tracks appearing later.

This feature allows track position to be compared more consistently across releases of different sizes.


In [23]:
df_model["relative_track_position"] = (
    df_model["track_number"] / df_model["album_total_tracks"]
)

In [24]:
df_model["relative_track_position"].describe()

count    8775.000000
mean        0.547097
std         0.341961
min         0.005525
25%         0.230769
50%         0.500000
75%         0.937500
max         1.000000
Name: relative_track_position, dtype: float64

In [25]:
invalid_positions = df_model[
    (df_model["relative_track_position"] <= 0) |
    (df_model["relative_track_position"] > 1)
]

print("Invalid relative positions:", len(invalid_positions))

Invalid relative positions: 0


### 5.2 Release Period

### Purpose

Create an interpretable categorical representation of when a track was released.

The dataset already contains `release_year`, which provides precise numerical information about release timing. A broader `release_decade` feature will also be created to represent major release periods and allow the models to capture differences between musical eras without assuming that the relationship between release year and popularity is strictly linear.

Both variables will initially be retained for analysis. However, they represent closely related information and will not automatically be included together in every model.

The final feature selection will consider redundancy and model interpretability before predictive modelling.


In [26]:
df_model["release_decade"] = (
    (df_model["release_year"] // 10) * 10
).astype(str) + "s"

In [27]:
df_model["release_decade"].value_counts().sort_index()

release_decade
1950s       7
1960s      50
1970s      80
1980s     100
1990s     406
2000s     937
2010s    3827
2020s    3368
Name: count, dtype: int64

In [28]:
print("Earliest release year:", df_model["release_year"].min())
print("Latest release year:", df_model["release_year"].max())
print("Release decades:", sorted(df_model["release_decade"].unique()))

Earliest release year: 1952
Latest release year: 2025
Release decades: ['1950s', '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s']


### 5.3 Transform Artist Followers

### Purpose

Prepare `artist_followers` for use in the full-information benchmark model.

Follower counts can vary substantially between artists, with a small number of highly followed artists having values far above the majority of observations. This type of right-skewed distribution can reduce interpretability and influence models that are sensitive to feature scale and extreme values.

A logarithmic transformation will therefore be created using `log1p`, which calculates:

**log(1 + artist_followers)**

The `+1` allows the transformation to remain valid even when a follower count is zero.

The original `artist_followers` variable will be retained for reference, while the transformed feature will be considered for Model B.


In [29]:
df_model["artist_followers"].describe()

count    8.772000e+03
mean     2.437636e+07
std      3.816164e+07
min      0.000000e+00
25%      5.160382e+05
50%      6.294820e+06
75%      3.055207e+07
max      1.455421e+08
Name: artist_followers, dtype: float64

In [30]:
df_model["log_artist_followers"] = np.log1p(
    df_model["artist_followers"]
)

In [31]:
df_model["log_artist_followers"].describe()

count    8772.000000
mean       14.817919
std         3.227032
min         0.000000
25%        13.153937
50%        15.655238
75%        17.234943
max        18.795976
Name: log_artist_followers, dtype: float64

In [32]:
print(
    "Missing original followers:",
    df_model["artist_followers"].isna().sum()
)

print(
    "Missing log followers:",
    df_model["log_artist_followers"].isna().sum()
)

Missing original followers: 3
Missing log followers: 3


## 6. Create the High-Popularity Classification Target

### Purpose

Create a binary target variable for the classification component of the machine learning analysis.

The regression analysis will predict the continuous `track_popularity` score. The classification analysis will address a related but distinct question: whether a track belongs to the high-popularity segment of the dataset.

Rather than selecting an arbitrary popularity score, the classification threshold will be determined from the observed distribution of `track_popularity`.

Tracks at or above the 75th percentile will be classified as **high popularity**, while tracks below the threshold will be classified as **lower popularity**.

This percentile-based approach provides a reproducible definition based on the actual dataset and is designed to identify approximately the top quarter of tracks by popularity.


In [33]:
popularity_threshold = df_model["track_popularity"].quantile(0.75)

print("75th percentile popularity threshold:", popularity_threshold)

75th percentile popularity threshold: 71.0


In [34]:
print(
    "Tracks below threshold:",
    (df_model["track_popularity"] < popularity_threshold).sum()
)

print(
    "Tracks at or above threshold:",
    (df_model["track_popularity"] >= popularity_threshold).sum()
)

Tracks below threshold: 6512
Tracks at or above threshold: 2263


### 6.1 Final Classification Threshold

The 75th percentile of `track_popularity` is **71**.

Using this threshold:

* 6,512 tracks fall below 71
* 2,263 tracks have a popularity score of 71 or higher

This produces a high-popularity class representing approximately 25.8% of the dataset.

The threshold will therefore be retained because it closely represents the intended top-quarter segment while preserving a simple and reproducible classification rule.

The binary target will be defined as:

* `1` = High popularity (`track_popularity >= 71`)
* `0` = Lower popularity (`track_popularity < 71`)

Because multiple tracks share the threshold score of 71, the high-popularity group is slightly larger than exactly 25% of the dataset.


In [35]:
df_model["high_popularity"] = (
    df_model["track_popularity"] >= popularity_threshold
).astype(int)

In [36]:
df_model["high_popularity"].value_counts()

high_popularity
0    6512
1    2263
Name: count, dtype: int64

In [37]:
df_model["high_popularity"].value_counts(normalize=True).mul(100).round(2)

high_popularity
0    74.21
1    25.79
Name: proportion, dtype: float64

## 7. Define Model Feature Sets

### Purpose

Define the predictor variables that will be used in the two machine learning feature sets.

Two feature sets are maintained to separate the predictive value of intrinsic track and release characteristics from information about the artist's established market position.

### Model A: Intrinsic Track and Release Features

Model A represents the primary business model. It uses characteristics that describe the track or its release without relying on artist popularity or follower count.

The predictors are:

**Numerical features**

* `track_duration_min`
* `album_total_tracks`
* `relative_track_position`

**Categorical features**

* `explicit`
* `genre_grouped`
* `album_type`
* `release_decade`

This model addresses whether track popularity can be predicted using intrinsic track and release characteristics.

### Model B: Full Information Benchmark

Model B uses all Model A predictors and additionally includes:

* `artist_popularity`
* `log_artist_followers`

This benchmark measures how much predictive performance changes when information about the artist's established market position is available.

### Feature Selection Decisions

`track_number` is replaced by `relative_track_position` because relative position provides a more comparable measure across releases of different sizes.

`artist_genres` and `primary_genre` are replaced by the simplified `genre_grouped` feature.

`artist_followers` is represented by `log_artist_followers` to reduce the influence of its strongly skewed distribution.

`release_year` is retained in the feature-engineered dataset for reference, but `release_decade` will be used in the initial models to allow differences between release periods without assuming a strictly linear relationship with popularity.

Identifiers and descriptive names are excluded from the predictor sets.


In [38]:
model_a_numeric = [
    "track_duration_min",
    "album_total_tracks",
    "relative_track_position"
]

model_a_categorical = [
    "explicit",
    "genre_grouped",
    "album_type",
    "release_decade"
]

model_b_numeric = model_a_numeric + [
    "artist_popularity",
    "log_artist_followers"
]

model_b_categorical = model_a_categorical.copy()

In [39]:
model_a_features = model_a_numeric + model_a_categorical
model_b_features = model_b_numeric + model_b_categorical

print("Model A features:")
print(model_a_features)

print("\nModel B features:")
print(model_b_features)

Model A features:
['track_duration_min', 'album_total_tracks', 'relative_track_position', 'explicit', 'genre_grouped', 'album_type', 'release_decade']

Model B features:
['track_duration_min', 'album_total_tracks', 'relative_track_position', 'artist_popularity', 'log_artist_followers', 'explicit', 'genre_grouped', 'album_type', 'release_decade']


## 8. Validate Engineered Features

### Purpose

Perform a final quality check on the engineered features before preprocessing and machine learning.

This validation confirms that:

* All required Model A and Model B features exist
* Missing values are identified
* Numerical features contain valid finite values
* Categorical features contain valid categories
* Regression and classification targets are correctly defined
* Target variables are not included in the predictor sets

Model A should contain complete intrinsic track and release information.

Model B may retain the small number of missing artist-level observations identified earlier. These missing values will be handled during the preprocessing stage rather than manually modifying the original observations.


In [40]:
print("Model A missing values:")
print(df_model[model_a_features].isna().sum())

print("\nModel B missing values:")
print(df_model[model_b_features].isna().sum())

Model A missing values:
track_duration_min         0
album_total_tracks         0
relative_track_position    0
explicit                   0
genre_grouped              0
album_type                 0
release_decade             0
dtype: int64

Model B missing values:
track_duration_min         0
album_total_tracks         0
relative_track_position    0
artist_popularity          3
log_artist_followers       3
explicit                   0
genre_grouped              0
album_type                 0
release_decade             0
dtype: int64


In [41]:
numeric_features = list(
    dict.fromkeys(model_a_numeric + model_b_numeric)
)

df_model[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
track_duration_min,8775.0,3.503838,1.053673,0.150000,2.900000,3.450000,4.000000,13.520000
album_total_tracks,8775.0,13.783362,11.798278,1.000000,6.000000,13.000000,17.000000,181.000000
relative_track_position,8775.0,0.547097,0.341961,0.005525,0.230769,0.500000,0.937500,1.000000
artist_popularity,8772.0,69.980506,19.500541,0.000000,60.000000,74.000000,84.000000,100.000000
log_artist_followers,8772.0,14.817919,3.227032,0.000000,13.153937,15.655238,17.234943,18.795976


In [42]:
print(
    "Infinite numerical values:",
    np.isinf(df_model[numeric_features]).sum().sum()
)

Infinite numerical values: 0


In [43]:
targets = ["track_popularity", "high_popularity"]

print(
    "Targets in Model A:",
    [col for col in targets if col in model_a_features]
)

print(
    "Targets in Model B:",
    [col for col in targets if col in model_b_features]
)

Targets in Model A: []
Targets in Model B: []


## 9. Save the Feature-Engineered Dataset

### Purpose

Save the completed feature-engineered dataset for use in the machine learning stage.

The saved dataset retains the original cleaned variables together with the engineered features created in this notebook. This provides a reproducible transition between data preparation and predictive modelling.

The main engineered variables include:

* `primary_genre`
* `genre_grouped`
* `relative_track_position`
* `release_decade`
* `log_artist_followers`
* `high_popularity`

The original variables are retained where useful for reference, validation, and model interpretation.

Model-specific preprocessing, including categorical encoding, numerical scaling, and treatment of the small number of missing artist-level values, will be performed during the machine learning stage. This avoids modifying the analytical dataset specifically for one model.

The resulting dataset will support:

* Model A using intrinsic track and release characteristics
* Model B using the same features plus artist popularity information
* Regression using `track_popularity`
* Classification using `high_popularity`


In [44]:
print("Final shape:", df_model.shape)

print("\nEngineered columns:")
engineered_columns = [
    "primary_genre",
    "genre_grouped",
    "relative_track_position",
    "release_decade",
    "log_artist_followers",
    "high_popularity"
]

print(engineered_columns)

df_model[engineered_columns].head()

Final shape: (8775, 22)

Engineered columns:
['primary_genre', 'genre_grouped', 'relative_track_position', 'release_decade', 'log_artist_followers', 'high_popularity']


,primary_genre,genre_grouped,relative_track_position,release_decade,log_artist_followers,high_popularity
0,pop,pop,0.982759,2000s,16.692203,0
1,stutter house,Other,1.000000,2020s,12.590433,0
2,Unknown,Unknown,0.423077,2020s,18.794974,0
3,Unknown,Unknown,0.125000,2020s,18.592044,1
4,Unknown,Unknown,0.538462,2010s,18.049576,0


In [45]:
output_path = "data/clean/spotify_features.csv"

df_model.to_csv(output_path, index=False)

print(f"Feature-engineered dataset saved to: {output_path}")

Feature-engineered dataset saved to: data/clean/spotify_features.csv


In [46]:
df_check = pd.read_csv("data/clean/spotify_features.csv")

print("Saved dataset shape:", df_check.shape)
print("File successfully verified:", df_check.shape == df_model.shape)

Saved dataset shape: (8775, 22)
File successfully verified: True


## 10. Feature Engineering Summary

The feature engineering stage prepared the cleaned Spotify dataset for regression and classification modelling.

### Key Transformations

The following features were created:

* `primary_genre` simplified the original multi-label genre information by retaining the first listed genre.
* `genre_grouped` reduced sparse genre categories by grouping primary genres with fewer than 30 observations into `Other`, while preserving `Unknown` as a separate category.
* `relative_track_position` standardized track position according to the total number of tracks on each release.
* `release_decade` created an interpretable categorical representation of release period.
* `log_artist_followers` reduced the strong skew in artist follower counts.
* `high_popularity` created the binary classification target using the 75th percentile of track popularity.

### Classification Target

The 75th percentile of `track_popularity` was 71.

Tracks were therefore classified as:

* `1` = High popularity, with `track_popularity >= 71`
* `0` = Lower popularity, with `track_popularity < 71`

The resulting class distribution was:

* Lower popularity: 6,512 tracks, or 74.21%
* High popularity: 2,263 tracks, or 25.79%

### Model Feature Sets

**Model A** uses intrinsic track and release characteristics:

* Track duration
* Album size
* Relative track position
* Explicit content
* Grouped genre
* Album type
* Release decade

**Model B** uses the same features and additionally includes:

* Artist popularity
* Log-transformed artist followers

Model B will serve as the full-information benchmark for evaluating how much predictive performance changes when artist-level popularity information is available.

### Data Quality

Final validation confirmed that:

* Model A contains no missing predictor values.
* Model B contains only three missing observations in the artist-level variables.
* No infinite numerical values are present.
* Neither regression nor classification targets are included in the predictor sets.
* The feature-engineered dataset contains 8,775 observations and 22 variables.

The completed dataset was saved as:

`data/clean/spotify_features.csv`

The next stage will apply preprocessing pipelines and train regression and classification models using the defined Model A and Model B feature sets.
